# Tips 01：Excel売上データを自動集計

**業務の困りごと**：Excelで担当者別・商品別の売上を手作業で集計する。

**After**：Colabでサンプルを作り、処理して結果を確認します。

**使用ライブラリ**：pandas, openpyxl

### ファイルの場所
`/content/` はColabの一時領域です。PCやGoogle Driveには自動保存されません。左側のフォルダから結果を選んでダウンロードできます。セッション終了前に保存してください。

上から順番に実行してください。サンプルは各ノートブック内で作ります。再実行時は同名の練習ファイルを上書きします。


## 1. 練習用データと準備
サンプルを生成し、`head()` で先頭の内容を確認します。実務では、この入力を自分のファイルに置き換えます。


In [ ]:
import pandas as pd
from pathlib import Path
base = Path('/content') if Path('/content').exists() else Path.cwd()
src = base / 'sample_sales.xlsx'
pd.DataFrame({'日付':['2026-09-01','2026-09-02','2026-09-03','2026-09-04'], '店舗':['鹿児島','霧島','鹿児島','霧島'], '担当者':['田中','佐藤','佐藤','田中'], '商品':['茶','菓子','茶','菓子'], '売上金額':[1200,800,1500,900]}).to_excel(src,index=False)
print('練習用ファイル:', src)
print('練習用データ（先頭5行）:')
pd.read_excel(src).head()


## 2. 自動処理
入力を読み込み、目的の形式で保存します。


In [ ]:
df = pd.read_excel(src)
df['日付'] = pd.to_datetime(df['日付'])
by_staff = df.groupby('担当者',as_index=False)['売上金額'].sum()
by_product = df.groupby('商品',as_index=False)['売上金額'].sum()
by_month = df.assign(月=df['日付'].dt.strftime('%Y-%m')).groupby('月',as_index=False)['売上金額'].sum()
out = base/'Tips01_売上集計.xlsx'
with pd.ExcelWriter(out) as writer:
    df.to_excel(writer,sheet_name='元データ',index=False)
    by_staff.to_excel(writer,sheet_name='担当者別',index=False)
    by_product.to_excel(writer,sheet_name='商品別',index=False)
    by_month.to_excel(writer,sheet_name='月別',index=False)
print(by_staff, '\n', by_product)


## 3. 件数・内容の確認
入力と出力が意図した内容か確認します。


In [ ]:
assert by_staff['売上金額'].sum() == df['売上金額'].sum() == 4400
assert out.exists()
print('確認OK:', out)


## 4. 持ち帰り・練習

**練習問題**：「店舗別」シートも追加してみましょう。



出力をPCへ保存したい場合は次のセルをColabで実行します。Colab以外では出力場所を表示します。


In [ ]:
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print('出力ファイル:',out.resolve())
